# dbt 3: Incremental models and microbatch

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/dbt/dbt-03-incremental-models.ipynb)

Run dbt locally against DuckDB using a fresh project for each lesson.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q dbt-core~=1.9 dbt-duckdb~=1.9 duckdb~=2.0

## Setup and sample data

Create a small dbt project and database in a temporary directory.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import duckdb
from dbt.cli.main import dbtRunner

workspace = TemporaryDirectory()
project = Path(workspace.name)
models = project / "models"
models.mkdir()
warehouse = project / "warehouse.duckdb"
(project / "dbt_project.yml").write_text("name: handbook\nversion: '1.0'\nprofile: handbook\n")
(project / "profiles.yml").write_text(
    f"handbook:\n  outputs:\n    dev:\n      type: duckdb\n      path: '{warehouse}'\n      threads: 1\n"
    "  target: dev\n"
)


def dbt(*args, expect_success=True):
    result = dbtRunner().invoke([*args, "--project-dir", str(project), "--profiles-dir", str(project)])
    if result.success != expect_success:
        raise RuntimeError(f"Unexpected dbt result: {result.exception or result.result}")
    print("dbt completed:", result.success)


def query(sql):
    with duckdb.connect(str(warehouse)) as connection:
        print(connection.execute(sql).fetchall())

## Incremental models and microbatch

An incremental model can limit the input it processes after the first build. A merge updates matching keys instead of inserting duplicates.

**Where it matters**

Keep an event or source sequence when a change can arrive late. Use event-time microbatching when bounded intervals are independently rebuildable.

**Watch out for**

A global high-water mark can miss late events. Define a lookback or replay strategy. Verify adapter support before choosing a microbatch strategy.

In [ ]:
with duckdb.connect(str(warehouse)) as connection:
    connection.execute("create table incoming(id integer, cents integer, seq integer)")
    connection.execute("insert into incoming values (1, 1200, 1)")
(models / "orders.sql").write_text("""
{{ config(materialized='incremental', unique_key='id', incremental_strategy='merge') }}
select * from incoming
{% if is_incremental() %}
where seq > (select coalesce(max(seq), 0) from {{ this }})
{% endif %}
qualify row_number() over (partition by id order by seq desc) = 1
""")
dbt("run")
with duckdb.connect(str(warehouse)) as connection:
    connection.execute("insert into incoming values (1, 1400, 2), (2, 800, 3)")
dbt("run")
dbt("run")
query("select * from orders order by id")

**What to notice:** The rerun leaves rows (1, 1400, 2) and (2, 800, 3). The first order was updated by key.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

Would an unseen event with sequence 2 pass this filter after sequence 3 has loaded?

In [ ]:
# Try your solution here.

In [ ]:
print("No. This filter assumes new input arrives above the current maximum.")

## Key takeaways

- Keep an event or source sequence when a change can arrive late. Use event-time microbatching when bounded intervals are independently rebuildable.

**Next:** [dbt 4](dbt-04-snapshots.ipynb)